In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

os.listdir()

In [ ]:
customers = pd.read_csv("Customers.csv")
orders = pd.read_csv("Orders.csv")
order_items = pd.read_csv("Order_Items.csv")
products = pd.read_csv("Products.csv")
returns = pd.read_csv("Returns.csv")
reviews = pd.read_csv("Reviews.csv")

In [ ]:
print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)
print("Returns:", returns.shape)
print("Reviews:", reviews.shape)

In [ ]:
print("CUSTOMERS")
print(customers.columns.tolist())

print("\nORDERS")
print(orders.columns.tolist())

print("\nORDER ITEMS")
print(order_items.columns.tolist())

print("\nPRODUCTS")
print(products.columns.tolist())

print("\nRETURNS")
print(returns.columns.tolist())

print("\nREVIEWS")
print(reviews.columns.tolist())

For this project analysis I do it into 8 main sections:
1. Data Cleaning & Quality Checks
2. Overall KPIs
3. Sales Trend Analysis
4. Product Analysis
5. Customer & Acquisition Analysis
6. Returns & Reviews Analysis
7. Profitability Analysis
8. Revenue Validation

**1. Data Quality Check**

In [ ]:
print("CUSTOMERS")
print(customers.isnull().sum())

print("\nORDERS")
print(orders.isnull().sum())

print("\nORDER ITEMS")
print(order_items.isnull().sum())

print("\nPRODUCTS")
print(products.isnull().sum())

print("\nRETURNS")
print(returns.isnull().sum())

print("\nREVIEWS")
print(reviews.isnull().sum())

Investigate delivered_date

In [ ]:
orders["order_status"].value_counts()

In [ ]:
pd.crosstab(
    orders["order_status"],
    orders["delivered_date"].isna()
)

FINDING IF ANY DUPLICATE VALUES PRESENT

In [ ]:
print("Customers duplicates:", customers.duplicated().sum())
print("Orders duplicates:", orders.duplicated().sum())
print("Order Items duplicates:", order_items.duplicated().sum())
print("Products duplicates:", products.duplicated().sum())
print("Returns duplicates:", returns.duplicated().sum())
print("Reviews duplicates:", reviews.duplicated().sum())

In [ ]:
print("Customer IDs:", customers["customer_id"].nunique(), "/", len(customers))
print("Order IDs:", orders["order_id"].nunique(), "/", len(orders))
print("Order Item IDs:", order_items["order_item_id"].nunique(), "/", len(order_items))
print("Product IDs:", products["product_id"].nunique(), "/", len(products))
print("Return IDs:", returns["return_id"].nunique(), "/", len(returns))
print("Review IDs:", reviews["review_id"].nunique(), "/", len(reviews))

Checking data types

In [ ]:
print("CUSTOMERS")
customers.info()

print("\nORDERS")
orders.info()

print("\nORDER ITEMS")
order_items.info()

print("\nPRODUCTS")
products.info()

print("\nRETURNS")
returns.info()

print("\nREVIEWS")
reviews.info()

In [ ]:
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"], format="%d-%m-%Y", errors="coerce"
)

orders["order_date"] = pd.to_datetime(
    orders["order_date"], format="%d-%m-%Y", errors="coerce"
)

orders["delivered_date"] = pd.to_datetime(
    orders["delivered_date"], format="%d-%m-%Y", errors="coerce"
)

products["launch_date"] = pd.to_datetime(
    products["launch_date"], format="%d-%m-%Y", errors="coerce"
)

returns["return_date"] = pd.to_datetime(
    returns["return_date"], format="%d-%m-%Y", errors="coerce"
)

reviews["review_date"] = pd.to_datetime(
    reviews["review_date"], format="%d-%m-%Y", errors="coerce"
)

In [ ]:
orders.dtypes

Checking the abnormal values

In [ ]:
print("Orders:")
print("Negative gross amount:", (orders["gross_amount"] < 0).sum())
print("Negative discount:", (orders["discount_amount"] < 0).sum())
print("Negative shipping fee:", (orders["shipping_fee"] < 0).sum())
print("Negative final amount:", (orders["final_amount"] < 0).sum())

print("\nOrder Items:")
print("Negative quantity:", (order_items["quantity"] < 0).sum())
print("Negative unit price:", (order_items["unit_price"] < 0).sum())
print("Invalid discount %:", ((order_items["discount_pct"] < 0) |
                              (order_items["discount_pct"] > 100)).sum())

print("\nProducts:")
print("Negative MRP:", (products["mrp"] < 0).sum())
print("Negative cost price:", (products["cost_price"] < 0).sum())
print("Negative stock:", (products["stock_qty"] < 0).sum())

print("\nReviews:")
print("Invalid ratings:", ((reviews["rating"] < 1) |
                           (reviews["rating"] > 5)).sum())

EDA

9.1 — Understand categorical values

In [ ]:
print("Order Status:")
print(orders["order_status"].value_counts())

print("\nPayment Methods:")
print(orders["payment_method"].value_counts())

print("\nSales Channels:")
print(orders["sales_channel"].value_counts())

print("\nAcquisition Channels:")
print(customers["acquisition_channel"].value_counts())

print("\nProduct Categories:")
print(products["category"].value_counts())

print("\nSkin Types:")
print(products["skin_type"].value_counts())

print("\nRatings:")
print(reviews["rating"].value_counts().sort_index())


**2. KPIs**

In [ ]:
# Overall KPIs

total_revenue = order_items["item_total"].sum()

total_orders = orders["order_id"].nunique()

total_customers = customers["customer_id"].nunique()

average_order_value = total_revenue / total_orders

print(f"Total Revenue: ₹{total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Average Order Value: ₹{average_order_value:,.2f}")

### Revenue Definition

Product revenue is calculated using `Order_Items[item_total]`, which represents revenue at the product/order-item level. This metric is used as the primary revenue measure throughout the analysis.


In [ ]:
monthly_sales = (
    order_items
    .merge(
        orders[["order_id", "order_date"]],
        on="order_id",
        how="left"
    )
    .groupby(order_items.merge(
        orders[["order_id", "order_date"]],
        on="order_id",
        how="left"
    )["order_date"].dt.to_period("M"))["item_total"]
    .sum()
    .reset_index(name="revenue")
)

monthly_sales["order_date"] = monthly_sales["order_date"].astype(str)
monthly_sales


Monthly orders data

In [ ]:
monthly_orders = (
    orders
    .groupby(orders["order_date"].dt.to_period("M"))["order_id"]
    .nunique()
    .reset_index(name="orders")

)

monthly_orders["order_date"] = monthly_orders["order_date"].astype(str)
monthly_orders




Monthly AOV (Average Order Value).

In [ ]:
monthly_order_data = order_items.merge(
    orders[["order_id", "order_date"]],
    on="order_id",
    how="left"
)

monthly_aov = (
    monthly_order_data
    .groupby(monthly_order_data["order_date"].dt.to_period("M"))
    .agg(
        orders=("order_id", "nunique"),
        revenue=("item_total", "sum")
    )
    .reset_index()
)

monthly_aov["AOV"] = monthly_aov["revenue"] / monthly_aov["orders"]

monthly_aov["order_date"] = monthly_aov["order_date"].astype(str)

monthly_aov


We'll combine:

Orders → Order Items → Products

and calculate revenue by product category.

In [ ]:
category_sales = (
    order_items
    .merge(products[["product_id","category"]],on="product_id" , how="left")
    .groupby("category")["item_total"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()

)

category_sales

**3. Product Analysis**

Quantity vs Revenue

Here I find out whether Serum's dominance comes from selling more units or from having higher-priced products.

In [ ]:
category_performance = (
    order_items
    .merge(
        products[["product_id", "category"]],
        on="product_id",
        how="left"
    )
    .groupby("category")
    .agg(
        revenue=("item_total", "sum"),
        units_sold=("quantity", "sum")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

category_performance

We have 9 serum products, so the next is:

1.Which specific serum products are driving those 919 units?

In [ ]:
serum_products = (
    order_items
    .merge(
        products[["product_id", "product_name", "category"]],
        on="product_id",
        how="left"
    )
)

serum_products = (
    serum_products[serum_products["category"] == "Serum"]
    .groupby(["product_id", "product_name"])
    .agg(
        revenue=("item_total", "sum"),
        units_sold=("quantity", "sum")
    )
    .reset_index()
    .sort_values("revenue", ascending=False)
)

serum_products

2.How many customers are actually coming back and purchasing again?

In [ ]:
customer_orders = (
    orders
    .groupby("customer_id")["order_id"]
    .nunique()
)

repeat_customers = (customer_orders > 1).sum()
one_time_customers = (customer_orders == 1).sum()

print("One-time customers:", one_time_customers)
print("Repeat customers:", repeat_customers)
print("Repeat customer rate:",
      f"{repeat_customers / len(customer_orders) * 100:.2f}%")

3.Why are some registered customers not converting into customers with orders?

In [ ]:
customers_with_orders = orders["customer_id"].nunique()

customers_without_orders = (
    customers["customer_id"].nunique()
    - customers_with_orders
)

print("Customers with orders:", customers_with_orders)
print("Customers without orders:", customers_without_orders)

**5. Acquisition channel conversion**

In [ ]:
customer_order_status = customers[["customer_id", "acquisition_channel"]].copy()

customer_order_status["has_order"] = (
    customer_order_status["customer_id"].isin(orders["customer_id"])
)

channel_conversion = (
    customer_order_status
    .groupby("acquisition_channel")
    .agg(
        total_customers=("customer_id", "count"),
        customers_with_orders=("has_order", "sum")
    )
    .reset_index()
)

channel_conversion["conversion_rate"] = (
    channel_conversion["customers_with_orders"]
    / channel_conversion["total_customers"]
    * 100
)

channel_conversion.sort_values("conversion_rate", ascending=False)

Revenue by acquisition channel

In [ ]:
channel_order_items = (
    order_items
    .merge(
        orders[["order_id", "customer_id"]],
        on="order_id",
        how="left"
    )
    .merge(
        customers[["customer_id", "acquisition_channel"]],
        on="customer_id",
        how="left"
    )
)

channel_revenue = (
    channel_order_items
    .groupby("acquisition_channel")
    .agg(
        revenue=("item_total", "sum"),
        orders=("order_id", "nunique"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
)

channel_revenue["AOV"] = (
    channel_revenue["revenue"] / channel_revenue["orders"]
)

channel_revenue.sort_values("revenue", ascending=False)


AOV Conclusion

I analyzed acquisition channels using customer conversion, revenue, order volume, and AOV. Google Search generated the highest revenue, while Website Direct had the highest average order value. Referral had a high conversion rate but a comparatively smaller revenue contribution, showing that channel performance varies depending on the metric being evaluated.

**6. Returns Analysis**

In [ ]:
total_orders = orders["order_id"].nunique()

returned_orders = orders.loc[
    orders["order_status"] == "Returned",
    "order_id"
].nunique()

return_rate = returned_orders / total_orders * 100

print("Total Orders:", total_orders)
print("Returned Orders:", returned_orders)
print(f"Return Rate: {return_rate:.2f}%")

6.1 Return Reasons

In [ ]:
return_reasons = (
    returns["return_reason"]
    .value_counts()
    .reset_index()
)

return_reasons.columns = ["return_reason", "return_count"]

return_reasons

6.2 Which products are being returned?

In [ ]:
return_products = (
    returns
    .merge(
        products[["product_id", "product_name", "category"]],
        on="product_id",
        how="left"
    )
    .groupby(["product_id", "product_name", "category"])
    .size()
    .reset_index(name="return_count")
    .sort_values("return_count", ascending=False)
)

return_products

6.3 Product return rate

Let's calculate returns ÷ units sold.

In [ ]:
product_return_rate = (
    order_items
    .groupby("product_id")
    .agg(
        units_sold=("quantity", "sum")
    )
    .reset_index()
)

product_return_rate = product_return_rate.merge(
    return_products[["product_id", "product_name", "category", "return_count"]],
    on="product_id",
    how="left"
)

product_return_rate["return_count"] = (
    product_return_rate["return_count"].fillna(0)
)

product_return_rate["return_rate"] = (
    product_return_rate["return_count"]
    / product_return_rate["units_sold"]
    * 100
)

product_return_rate.sort_values(
    "return_rate",
    ascending=False
)


A products with lower customer ratings also have higher return rates ?

so i already have Reviews.csv, so i analyze it.

In [ ]:
product_ratings = (
    reviews
    .groupby("product_id")
    .agg(
        average_rating=("rating", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
)

product_analysis = product_return_rate.merge(
    product_ratings,
    on="product_id",
    how="left"
)

product_analysis.sort_values(
    "return_rate",
    ascending=False
)

Conclusion:
I compared product return rates with average customer ratings. Some higher-return products had relatively lower ratings, but the pattern was not consistent across all products, so I avoided concluding that ratings alone explain product returns.

In [ ]:
products[
    products["product_id"].isin(["P013", "P020", "P022"])
]

In [ ]:
print("Order Items:", order_items["product_id"].dtype)
print("Products:", products["product_id"].dtype)
print("Returns:", returns["product_id"].dtype)
print("Reviews:", reviews["product_id"].dtype)

In [ ]:
print("Order Items:", order_items["product_id"].unique())
print("Products:", products["product_id"].unique())

Rebuild the return-rate analysis

In [ ]:
product_return_rate = (
    order_items
    .groupby("product_id")
    .agg(
        units_sold=("quantity", "sum")
    )
    .reset_index()
)

product_return_rate = product_return_rate.merge(
    returns.groupby("product_id")
    .size()
    .reset_index(name="return_count"),
    on="product_id",
    how="left"
)

product_return_rate["return_count"] = (
    product_return_rate["return_count"].fillna(0)
)

product_return_rate = product_return_rate.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left"
)

product_return_rate["return_rate"] = (
    product_return_rate["return_count"]
    / product_return_rate["units_sold"]
    * 100
)

product_return_rate = product_return_rate.sort_values(
    "return_rate",
    ascending=False
)

product_return_rate

**7. Reviews Analysis**

Customer rating distribution

In [ ]:
rating_distribution = (
  reviews["rating"]
  .value_counts()
  .sort_index()
  .reset_index()
)

rating_distribution.columns=["rating","review_count"]
rating_distribution

Average Rating By Product

In [ ]:
product_ratings = (
    reviews
    .merge(products[["product_id", "product_name"]], on="product_id", how="left")
    .groupby(["product_id", "product_name"])
    .agg(
        average_rating=("rating", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
    .sort_values("average_rating", ascending=False)
)

product_ratings

Category Level Ratings

In [ ]:
category_ratings = (
    reviews
    .merge(products[["product_id", "category"]], on="product_id", how="left")
    .groupby("category")
    .agg(
        average_rating=("rating", "mean"),
        review_count=("review_id", "count")
    )
    .reset_index()
    .sort_values("average_rating", ascending=False)
)

category_ratings

7. Profitability Analysis

In [ ]:
order_items["calculated_total"] = (
    order_items["quantity"]
    * order_items["unit_price"]
    * (1 - order_items["discount_pct"] / 100)
)

order_items["difference"] = (
    order_items["item_total"] - order_items["calculated_total"]
)

order_items["difference"].describe()

Calculate product-level estimated profit

In [ ]:
profit_analysis = (
    order_items
    .merge(
        products[["product_id", "product_name", "category", "cost_price"]],
        on="product_id",
        how="left"
    )
)

profit_analysis["total_cost"] = (
    profit_analysis["quantity"] * profit_analysis["cost_price"]
)

profit_analysis["profit"] = (
    profit_analysis["item_total"] - profit_analysis["total_cost"]
)

product_profit = (
    profit_analysis
    .groupby(["product_id", "product_name", "category"])
    .agg(
        revenue=("item_total", "sum"),
        total_cost=("total_cost", "sum"),
        profit=("profit", "sum"),
        units_sold=("quantity", "sum")
    )
    .reset_index()
    .sort_values("profit", ascending=False)
)

product_profit

Profit Margin %

In [ ]:
product_profit["profit_margin"] = (
    product_profit["profit"] / product_profit["revenue"] * 100

)

product_profit.sort_values("profit_margin", ascending= False)

final profitability check

In [ ]:
total_revenue = product_profit["revenue"].sum()
total_cost = product_profit["total_cost"].sum()
total_profit = product_profit["profit"].sum()
overall_margin = total_profit / total_revenue * 100

print(f"Revenue: ₹{total_revenue:,.2f}")
print(f"Cost: ₹{total_cost:,.2f}")
print(f"Estimated Gross Profit: ₹{total_profit:,.2f}")
print(f"Estimated Gross Margin: {overall_margin:.2f}%")

Final Validation

In [ ]:
order_items_total = order_items["item_total"].sum()
orders_final_total = orders["final_amount"].sum()


difference = order_items_total - orders_final_total

print(f"Order Items Total: ₹{order_items_total:,.2f}")
print(f"Orders Final Amount: ₹{orders_final_total:,.2f}")
print(f"Difference : ₹{difference:,.2f}")


Step 1 — Check the order-level formula

In [55]:
orders["calculated_final"] = (
    orders["gross_amount"]
    - orders["discount_amount"]
    + orders["shipping_fee"]
)

orders["difference"] = (
    orders["final_amount"] - orders["calculated_final"]
)

orders["difference"].describe()

,difference
count,1250.000000
mean,35.518920
std,285.849542
min,-2689.900000
25%,22.450000
50%,69.800000
75%,134.750000
max,584.050000


### Revenue Validation Finding

`gross_amount` reconciles with the sum of `Order_Items[item_total]`, while `final_amount` does not consistently reconcile with gross amount, discounts, and shipping fees.

A total of 1,111 out of 1,250 orders (88.88%) show a mismatch. Therefore, `Order_Items[item_total]` is used as the primary product revenue metric for this analysis.

The discrepancy in `final_amount` is retained as a documented data-quality finding rather than being artificially corrected.

In [ ]:
order_item_totals = (
    order_items
    .groupby("order_id")["item_total"]
    .sum()
    .reset_index(name="item_total_sum")
)

order_check = orders[
    ["order_id", "gross_amount", "discount_amount", "shipping_fee", "final_amount"]
].merge(
    order_item_totals,
    on="order_id",
    how="left"
)

order_check["gross_vs_items"] = (
    order_check["gross_amount"] - order_check["item_total_sum"]
)

order_check["gross_vs_items"].describe()

In [ ]:
orders["expected_final"] = (
    orders["gross_amount"]
    - orders["discount_amount"]
    + orders["shipping_fee"]
)

orders["final_difference"] = (
    orders["final_amount"] - orders["expected_final"]
)

orders[["gross_amount", "discount_amount", "shipping_fee",
        "final_amount", "expected_final", "final_difference"]].head(10)

Checking How many orders have inconsistent final amounts.

In [ ]:
orders["difference_abs"] = orders["final_difference"].abs()

print("Rows with mismatch:",
      (orders["difference_abs"] > 0.01).sum())

print("Rows matching:",
      (orders["difference_abs"] <= 0.01).sum())

print("Mismatch percentage:",
      (orders["difference_abs"] > 0.01).mean() * 100)

In [ ]:
status_check = (
    orders
    .groupby("order_status")
    .agg(
        orders=("order_id", "count"),
        total_gross=("gross_amount", "sum"),
        total_final=("final_amount", "sum"),
        total_discount=("discount_amount", "sum"),
        total_shipping=("shipping_fee", "sum")
    )
    .reset_index()
)

status_check

calculate revenue by status at the item level.

In [ ]:
status_item_revenue = (
    order_items
    .merge(
        orders[["order_id", "order_status"]],
        on="order_id",
        how="left"
    )
    .groupby("order_status")
    .agg(
        item_revenue=("item_total", "sum"),
        units_sold=("quantity", "sum")
    )
    .reset_index()
)

status_item_revenue